# 01 — Validate parsed holdings

Parse the latest Stashaway statement, apply role tags from `book_config`,
and check that every parsed portfolio reconciles cleanly against the
weight-sum tolerance and resolves against the universe map.

**Inputs:** `data/statements/<latest>.pdf`

**Outputs:** a summary table; a per-portfolio holdings table; a list of any
unmapped tickers that need adding to `STASHAWAY_UNIVERSE`.


In [1]:
from pathlib import Path
import pandas as pd

from hailmary.allocation.book_config import ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
assert STATEMENT_PATH.exists(), f'Statement not found at {STATEMENT_PATH}'

## Parse the statement

In [2]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
print(f'Parsed {len(parsed)} portfolios from {STATEMENT_PATH.name}')
print(f'Statement date: {parsed[0].statement_date}')

Parsed 15 portfolios from 2026-04 StashAway Monthly Statement.pdf
Statement date: 2026-04-01


## Reconcile weights

Each portfolio's weights should sum to 1.0 ± 1e-4. If any fail, the parser
raised inside `parse_statement`; this cell is a final visual check.

In [3]:
rows = [
    {
        'name': pf.name,
        'currency': pf.currency,
        'total_value': pf.total_value,
        'n_holdings': len(pf.holdings),
        'weight_sum': sum(h.weight for h in pf.holdings),
    }
    for pf in parsed
]
summary = pd.DataFrame(rows).sort_values('total_value', ascending=False)
summary

,name,currency,total_value,n_holdings,weight_sum
12,Simple SGD,SGD,1952415.36,3,1.0
2,General Investing,USD,371548.08,19,1.0
3,Simple USD,USD,274197.85,3,1.0
13,General SRS,USD,101299.95,19,1.0
11,Guitsa,SGD,72752.28,3,1.0
14,Crypto,USD,54802.94,4,1.0
0,BlackRock,USD,39180.06,21,1.0
7,High Dividend Yield,USD,23860.69,2,1.0
4,Singapore Investing,SGD,19979.29,7,1.0
9,SG ETF,SGD,9864.54,2,1.0


## Apply role tags and resolve universe

`from_parsed` looks each holding up in `STASHAWAY_UNIVERSE`. An
`UnknownAssetError` here means a ticker needs adding to the map.

In [4]:
portfolios = []
missing_roles = []
for pf in parsed:
    roles = ROLES.get(pf.name)
    if roles is None:
        missing_roles.append(pf.name)
        continue
    portfolios.append(from_parsed(pf, roles=roles))

if missing_roles:
    print('Portfolios with no role tag (edit book_config.ROLES):')
    for n in missing_roles:
        print(f'  - {n!r}')
else:
    print(f'All {len(portfolios)} portfolios resolved against the universe map.')

All 15 portfolios resolved against the universe map.


## Diagnostic vs hidden portfolios

In [5]:
diag_rows = [
    {
        'name': p.name,
        'roles': ','.join(sorted(r.value for r in p.roles)),
        'currency': p.currency,
        'total_value': p.total_value,
        'n_holdings': len(p.holdings),
    }
    for p in portfolios
]
diag = pd.DataFrame(diag_rows)
in_diag = diag[diag['roles'].str.contains('holding')]
hidden = diag[~diag['roles'].str.contains('holding')]
print(f'In diagnostic ({len(in_diag)}):')
display(in_diag.sort_values('total_value', ascending=False))
print(f'Hidden ({len(hidden)}):')
display(hidden)

In diagnostic (12):


,name,roles,currency,total_value,n_holdings
2,General Investing,"holding,managed_benchmark",USD,371548.08,19
13,General SRS,"holding,protected",USD,101299.95,19
14,Crypto,"custom,holding",USD,54802.94,4
0,BlackRock,"holding,managed_benchmark",USD,39180.06,21
7,High Dividend Yield,"custom,holding",USD,23860.69,2
4,Singapore Investing,"holding,managed_benchmark",SGD,19979.29,7
9,SG ETF,"custom,holding",SGD,9864.54,2
1,Energy,"custom,holding",USD,8220.98,2
10,Nasdaq Covered Call,"custom,holding",USD,7890.77,2
8,Ex-US Large-cap,"custom,holding",USD,7864.31,2


Hidden (3):


,name,roles,currency,total_value,n_holdings
3,Simple USD,protected,USD,274197.85,3
11,Guitsa,protected,SGD,72752.28,3
12,Simple SGD,protected,SGD,1952415.36,3


## Detailed holdings — first three portfolios

In [6]:
for p in portfolios[:3]:
    print(f'\n=== {p.name} ({p.currency}) — total {p.total_value:,.2f} ===')
    rows = [
        {
            'stashaway_id': h.stashaway_id,
            'yahoo_ticker': h.metadata.ticker,
            'asset_class': h.metadata.asset_class,
            'region': h.metadata.region,
            'sector': h.metadata.sector,
            'weight': h.weight,
            'value': h.value,
        }
        for h in p.holdings
    ]
    display(pd.DataFrame(rows))


=== BlackRock (USD) — total 39,180.06 ===


,stashaway_id,yahoo_ticker,asset_class,region,sector,weight,value
0,ISAC,ISAC.L,Equity,Global,Broad Market,0.047105,1845.57
1,IDTM,IDTM.L,Bond,US,Treasury 7-10Y,0.018209,713.44
2,IDTL,IDTL.L,Bond,US,Treasury 20+Y,0.038572,1511.24
3,CSUS,CSUS.L,Equity,US,Broad Market,0.173369,6792.62
4,IJPA,IJPA.L,Equity,Japan,Broad Market,0.039213,1536.36
5,ISFD,ISFD.L,Equity,UK,Broad Market,0.010563,413.87
6,CCAU,CCAU.L,Equity,Canada,Broad Market,0.031051,1216.57
7,IGLN,IGLN.L,Commodity,Global,Gold,0.025952,1016.79
8,ICHN,MCHI,Equity,China,Broad Market (proxy: MCHI),0.025628,1004.10
9,IMBS,IMBS.L,Bond,US,MBS,0.025622,1003.86



=== Energy (USD) — total 8,220.98 ===


,stashaway_id,yahoo_ticker,asset_class,region,sector,weight,value
0,XLE,XLE,Equity,US,Energy,0.999872,8219.93
1,CASH_USD,CASH_USD,Cash,US,Cash,0.000128,1.05



=== General Investing (USD) — total 371,548.08 ===


,stashaway_id,yahoo_ticker,asset_class,region,sector,weight,value
0,BBJP,BBJP,Equity,Japan,Broad Market,0.017233,6402.96
1,DXJ,DXJ,Equity,Japan,Hedged,0.043857,16294.84
2,FBTC,BTC-USD,Crypto,Global,Bitcoin (proxy: BTC-USD spot),0.058297,21660.01
3,FETH,ETH-USD,Crypto,Global,Ethereum (proxy: ETH-USD spot),0.057148,21233.09
4,FLIN,FLIN,Equity,India,Broad Market,0.025587,9506.92
5,GLDM,GLDM,Commodity,Global,Gold,0.052597,19542.28
6,ISAC,ISAC.L,Equity,Global,Broad Market,0.161761,60101.90
7,IVV,IVV,Equity,US,Broad Market,0.118243,43933.12
8,PPA,PPA,Equity,US,Aerospace & Defense,0.025876,9614.11
9,RSP,RSP,Equity,US,Broad Market,0.053951,20045.53


## Final assertion

If any of these fail, the rest of the diagnostic pipeline can't trust the
input — fix before running notebooks 02 and 03.

In [7]:
from hailmary.allocation.portfolios import Role

assert len(portfolios) == 15, f'Expected 15 portfolios, got {len(portfolios)}'
for p in portfolios:
    assert abs(sum(h.weight for h in p.holdings) - 1.0) < 1e-4, p.name
managed = [p for p in portfolios if Role.MANAGED_BENCHMARK in p.roles]
assert len(managed) >= 1, 'No MANAGED_BENCHMARK portfolios — diagnostic will skip benchmark deltas'
print('All checks passed — proceed to notebook 02 / 03.')

All checks passed — proceed to notebook 02 / 03.
